In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, TimeDistributed, Conv2D, MaxPooling2D, Flatten, LSTM, Dense, Dropout
import matplotlib.pyplot as plt

print("1. Generating Logical Synthetic Data (Moving Hand Simulation)...")
num_samples = 400       # عدد الفيديوهات
frames_per_video = 10   # عدد الإطارات في الفيديو الواحد
height, width = 32, 32  # حجم الإطار (صغير لسرعة التدريب)
channels = 1            # Grayscale
num_classes = 3

# إنشاء مصفوفات فارغة (شاشات سوداء)
X = np.zeros((num_samples, frames_per_video, height, width, channels), dtype=np.float32)
y = np.zeros((num_samples,), dtype=np.int32)

# رسم "المربع الأبيض" وتوجيه حركته بناءً على نوع الإيماءة
for i in range(num_samples):
    gesture = np.random.randint(0, num_classes)
    y[i] = gesture
    row = height // 2  # تثبيت الحركة في منتصف الشاشة عمودياً

    for t in range(frames_per_video):
        col = width // 2  # الافتراضي في المنتصف
        if gesture == 0:   # Swipe Right (حركة لليمين)
            col = int((t / frames_per_video) * width)
        elif gesture == 1: # Swipe Left (حركة لليسار)
            col = int((1 - t / frames_per_video) * width)
        # gesture == 2 هيفضل في النص زي ما هو

        # التأكد إن المربع مش هيخرج برا الشاشة
        col = min(width - 3, max(2, col))
        # رسم المربع
        X[i, t, row-2:row+2, col-2:col+2, 0] = 1.0

# One-Hot Encoding للتصنيفات
y_cat = tf.keras.utils.to_categorical(y, num_classes)

# تقسيم البيانات لتدريب واختبار
split_idx = int(0.8 * num_samples)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y_cat[:split_idx], y_cat[split_idx:]

print(f"Data ready! X_train shape: {X_train.shape}")

print("\n2. Building the CNN-LSTM Architecture...")
# تم استخدام Input(shape) لتجنب تحذيرات Keras الجديدة
model = Sequential([
    Input(shape=(frames_per_video, height, width, channels)),

    TimeDistributed(Conv2D(8, (3, 3), activation='relu', padding='same')),
    TimeDistributed(MaxPooling2D((2, 2))),

    TimeDistributed(Conv2D(16, (3, 3), activation='relu', padding='same')),
    TimeDistributed(MaxPooling2D((2, 2))),

    TimeDistributed(Flatten()),

    LSTM(32, activation='tanh'),
    Dropout(0.2),

    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("\n3. Training the Model (Watch the accuracy soar!)...")
history = model.fit(X_train, y_train, epochs=15, batch_size=16, validation_data=(X_test, y_test))

print("\n4. Evaluating...")
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Final Validation Accuracy: {acc * 100:.2f}%")

1. Generating Logical Synthetic Data (Moving Hand Simulation)...
Data ready! X_train shape: (320, 10, 32, 32, 1)

2. Building the CNN-LSTM Architecture...

3. Training the Model (Watch the accuracy soar!)...
Epoch 1/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 67ms/step - accuracy: 0.8656 - loss: 0.6455 - val_accuracy: 1.0000 - val_loss: 0.1858
Epoch 2/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 1.0000 - loss: 0.0793 - val_accuracy: 1.0000 - val_loss: 0.0110
Epoch 3/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.0108 - val_accuracy: 1.0000 - val_loss: 0.0034
Epoch 4/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.0045 - val_accuracy: 1.0000 - val_loss: 0.0021
Epoch 5/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 1.0000 - loss: 0.0036 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 6/15
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 0.0023 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 7/15
20/20 ━━━━━━━━━━━━━